# 02 — Data Cleaning (Lending Club)

Turn the 38-column raw selection into an analysis-ready, target-labeled dataset.
Every decision is conservative and documented — this notebook is the audit trail for
what was removed, encoded, or imputed, and why. No outcome information is used anywhere
in the cleaning logic; the target is built only at the end, from `loan_status`.

**Outcome:** `workspace.default.lending_club_cleaned` — 1,296,898 resolved loans,
zero nulls, target included.

**Cleaning decisions at a glance**

| Step | Decision | Scale |
| --- | --- | --- |
| String → numeric | 17 columns coerced with `errors='coerce'` (junk → NaN → dropped next step) | — |
| Unsalvageable rows | empty rows, unreported core fields, corrupted `emp_length` | 53,473 rows (2.37%) |
| `emp_length` | ordinal 0–10; null → **−1** (distinct from 0 = “< 1 year”) + missing flag | 143,793 nulls |
| `mths_since_last_delinq` / `_record` | has-history flag + median fill | 49.1% / 16.1% flagged |
| `tot_cur_bal` / `tot_coll_amt` | missing flag + median fill | 3.3% |
| Target | 1 = Charged Off / Default, 0 = Fully Paid; **unresolved statuses excluded** | 910,330 rows |

In [1]:
df_lending_club = spark.read.table('workspace.default.lending_club')

In [2]:
df_lending_club.display()

loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,acc_now_delinq,pub_rec_bankruptcies,tax_liens,mort_acc
3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,,7.0,0.0,2765.0,29.7,13.0,w,Individual,0.0,0.0,0.0,1.0
24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,small_business,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,,22.0,0.0,21470.0,19.2,38.0,w,Individual,0.0,0.0,0.0,4.0
20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,home_improvement,IL,10.78,0.0,Aug-2000,695.0,699.0,0.0,,,6.0,0.0,7869.0,56.2,18.0,w,Joint App,0.0,0.0,0.0,5.0
35000.0,35000.0,35000.0,60 months,14.85,829.9,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,debt_consolidation,NJ,17.06,0.0,Sep-2008,785.0,789.0,0.0,,,13.0,0.0,7802.0,11.6,17.0,w,Individual,0.0,0.0,0.0,1.0
10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,major_purchase,PA,25.37,1.0,Jun-1998,695.0,699.0,3.0,12.0,,12.0,0.0,21929.0,64.5,35.0,w,Individual,0.0,0.0,0.0,6.0
11950.0,11950.0,11950.0,36 months,13.44,405.18,C,C3,4 years,RENT,34000.0,Source Verified,Dec-2015,Fully Paid,debt_consolidation,GA,10.2,0.0,Oct-1987,690.0,694.0,0.0,,,5.0,0.0,8822.0,68.4,6.0,w,Individual,0.0,0.0,0.0,0.0
20000.0,20000.0,20000.0,36 months,9.17,637.58,B,B2,10+ years,MORTGAGE,180000.0,Not Verified,Dec-2015,Fully Paid,debt_consolidation,MN,14.67,0.0,Jun-1990,680.0,684.0,0.0,49.0,,12.0,0.0,87329.0,84.5,27.0,f,Individual,0.0,0.0,0.0,4.0
20000.0,20000.0,20000.0,36 months,8.49,631.26,B,B1,10+ years,MORTGAGE,85000.0,Not Verified,Dec-2015,Fully Paid,major_purchase,SC,17.61,1.0,Feb-1999,705.0,709.0,0.0,3.0,,8.0,0.0,826.0,5.7,15.0,w,Individual,0.0,0.0,0.0,3.0
10000.0,10000.0,10000.0,36 months,6.49,306.45,A,A2,6 years,RENT,85000.0,Not Verified,Dec-2015,Fully Paid,credit_card,PA,13.07,0.0,Apr-2002,685.0,689.0,1.0,,106.0,14.0,1.0,10464.0,34.5,23.0,w,Individual,0.0,1.0,0.0,1.0
8000.0,8000.0,8000.0,36 months,11.48,263.74,B,B5,10+ years,MORTGAGE,42000.0,Not Verified,Dec-2015,Fully Paid,credit_card,RI,34.8,0.0,Nov-1994,700.0,704.0,0.0,75.0,,8.0,0.0,7034.0,39.1,18.0,w,Individual,0.0,0.0,0.0,1.0


In [3]:
#print the shape of the data

print(f'rows: {df_lending_club.count()}', f'columns: {len(df_lending_club.columns)}')

rows: 2260701 columns: 38


In [4]:
#Stats summary

df_lending_club.describe().display()


summary,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,acc_now_delinq,pub_rec_bankruptcies,tax_liens,mort_acc
count,2260668,2260668,2260668,2260668,2260668,2260668,2260668,2260668,2113761,2260668,2260664,2260668,2260668,2260668,2260667,2260667,2258956,2260638,2260638,2260667,2260667,2260637,1102327,359347,2260558,2260576,2260614,2258834,2260618,2260651,2260613,2260480,2259064,2260363,2210663
mean,15046.931227849467,15041.664056818605,15023.437745306304,,13.092829114936396,445.8068228818726,,,,,77992.44637764223,38000.0,,,384.0,0.0,18.824907147958033,0.3072159491072534,24.6573786407767,698.5605010090696,702.5678527699907,0.6092147167778061,34.59148180842484,72.41191248510894,11.624544599950365,0.2062077129597312,16657.354743919248,50.86537411800748,24.46292690641181,2470.255207373272,884.8283570186857,0.02202342698543187,0.12824780528572896,0.04682079825231611,1.5553885870437965
stddev,9190.245488232804,9188.413022382023,9192.331678793584,,4.832138364571101,267.1735346084263,,,,,112696.22136091426,,,,543.0580079512685,,14.18377319930504,0.8703772261110874,197.67945675130858,33.28100152599884,33.21178665565171,4.9053651850447,22.71093973315836,27.814303730591018,6.351963115695965,2.4646275191778058,22948.04685404855,185.3175434907719,104.02520390697718,6979.857129762963,2697.380938769947,9.861136037608004,0.3706432865072621,0.3833605824898107,1.9052379160587243
min,500.0,500.0,0.0,36 months,5.31,4.93,A,A1,"reactors""",2 years,0.0,38000.0,Apr-2008,Charged Off,After graduating from college,American women use 12 personal care products daily. Seventy-nine percent apply a makeup product at least 22.3 times during an average week. An estimated<br/>90% of girls age 14 and older regularly use cosmetics.<br/> Borrower added on 03/28/10 > Business is good,"Karen """,BEST BUY,"Mike """,809,Capital One ($2768) and Discover($1490). I always pay more than the minimum payments,I am currently building the HUSH e-commerce website www.shopHUSH.com. I have made it a point to fund my business with personal capital and not by incurring debt and up until this point have been successful. However,"Carlos""","I plan on eliminating and decreasing the debt on cards #3 and 4<br/>""","I'm sure)!! Please help me get rid of this house! Thanks!<br/>""",#450216.<br/> 553609 added on 12/10/09 > Loan re-listed,and the work we do.<br/><br/>We are in the marketing research and consulting business. Our main task is to speak with customers,I am working on that as well. I do have about $30K in IRA's and have checking & savings set up. I am not in school so have no student loans,"I mean.<br/>""","and I pay her tuition up front each year in full.<br/><br/>Please let me know if I can answer any more questions for you! Thanks for your interest & hopefully I've convice you not to be a skeptic w/ me! - Ga Peach Girl<br/>""",0.0,0.0,0.0,0.0,0.0
max,40000.0,40000.0,40000.0,60 months,30.99,1719.83,G,G5,< 1 year,RENT,MORTGAGE,Verified,Source Verified,Oct-2015,wedding,wedding,wedding,small_business,small_business,small_business,wedding,small_business,home_improvement,home_improvement,debt_consolidation,moving,small_business,other,small_business,w,f,f,100.0,100.0,94.0


In [5]:
#Print schema

df_lending_club.printSchema()

root
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string (nullable = true)
 |-- delinq_2yrs: string (nullable = true)
 |-- earliest_cr_line: string (nullable = true)
 |-- fico_range_low: string (nullable = true)
 |-- fico_range_high: string (nullable = true)
 |-- inq_last_6mths: string (nullable = true)
 |-- mths_since_last_delinq: string (nullable = true)
 |--

In [6]:
#Convert to pandas

df_lending_club = df_lending_club.toPandas()
#Print first 5 rows

In [7]:
#Check null values

df_lending_club.isnull().sum()

loan_amnt                      33
funded_amnt                    33
funded_amnt_inv                33
term                           33
int_rate                       33
installment                    33
grade                          33
sub_grade                      33
emp_length                 146940
home_ownership                 33
annual_inc                     37
verification_status            33
issue_d                        33
loan_status                    33
purpose                        34
addr_state                     34
dti                          1745
delinq_2yrs                    63
earliest_cr_line               63
fico_range_low                 34
fico_range_high                34
inq_last_6mths                 64
mths_since_last_delinq    1158374
mths_since_last_record    1901354
open_acc                      143
pub_rec                       125
revol_bal                      87
revol_util                   1867
total_acc                      83
initial_list_s

## Cleaning and encoding

Rows that cannot be salvaged are dropped (2.37% — all missing bureau-reported values,
empty rows, or corrupted records); everything else is kept. Encodings use fixed,
documented mappings. Nulls in the “history” columns mean *no such history exists*, so
the absence itself becomes a feature flag and the numeric column is median-filled for
models that reject NaN (logistic regression).

In [8]:
import pandas as pd

# The source CSV stored many numbers as strings; convert them to real numerics.
# errors='coerce' turns any junk values into NaN, which the row-drop cell removes next.

numeric_cols = [
    'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'acc_now_delinq',
    'tot_cur_bal', 'tot_coll_amt', 'collections_12_mths_ex_med'
]

for col in numeric_cols:
    df_lending_club[col] = pd.to_numeric(df_lending_club[col], errors='coerce')

print(f'converted {len(numeric_cols)} columns to numeric')
df_lending_club[numeric_cols].dtypes

converted 17 columns to numeric


annual_inc                    float64
dti                           float64
delinq_2yrs                   float64
fico_range_low                float64
fico_range_high               float64
inq_last_6mths                float64
mths_since_last_delinq        float64
mths_since_last_record        float64
open_acc                      float64
pub_rec                       float64
revol_bal                     float64
revol_util                    float64
total_acc                     float64
acc_now_delinq                float64
tot_cur_bal                   float64
tot_coll_amt                  float64
collections_12_mths_ex_med    float64
dtype: object

In [9]:
# Drop rows that cannot be salvaged:
#   1) loan_amnt null               -> the 33 completely empty rows
#   2) nulls in unreported fields   -> revol_util, dti, pub_rec_bankruptcies, tax_liens, mort_acc
#                                      (genuinely missing bureau/financial values, ~53k rows / 2.4%)
#   3) nulls in core application fields -> a handful of partial rows each (fico, open_acc, ...)
#   4) emp_length not a valid category -> corrupted values (e.g. ' reactors"')

null_drop_cols = ['revol_util', 'dti', 'pub_rec_bankruptcies', 'tax_liens', 'mort_acc']
partial_null_cols = [
    'annual_inc', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'total_acc',
    'initial_list_status', 'application_type', 'acc_now_delinq', 'purpose', 'addr_state'
]
valid_emp_length = {
    '< 1 year', '1 year', '2 years', '3 years', '4 years', '5 years',
    '6 years', '7 years', '8 years', '9 years', '10+ years'
}

drop_mask = (
    df_lending_club['loan_amnt'].isnull()
    | df_lending_club[null_drop_cols].isnull().any(axis=1)
    | df_lending_club[partial_null_cols].isnull().any(axis=1)
    | (~df_lending_club['emp_length'].isin(valid_emp_length) & df_lending_club['emp_length'].notna())
)

print(f'rows before: {len(df_lending_club):,}')
df_lending_club = df_lending_club[~drop_mask].reset_index(drop=True)
print(f'rows dropped: {drop_mask.sum():,}')
print(f'rows after:  {len(df_lending_club):,}')

rows before: 2,260,701
rows dropped: 53,473
rows after:  2,207,228


In [10]:
# Map emp_length to an ordinal integer: '< 1 year' -> 0, '1 year' -> 1, ..., '10+ years' -> 10.
# Null (never reported) -> -1, kept DISTINCT from 0 so 'unknown' is not conflated with
# 'employed less than a year'; the companion flag lets models learn 'unknown' as its own effect.

emp_length_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
    '10+ years': 10
}

df_lending_club['emp_length_missing'] = df_lending_club['emp_length'].isnull().astype(int)
df_lending_club['emp_length'] = df_lending_club['emp_length'].map(emp_length_map).fillna(-1).astype(int)

print(df_lending_club['emp_length'].value_counts().sort_index())

emp_length
-1     143793
 0     184027
 1     144132
 2     198068
 3     175645
 4     132221
 5     135497
 6      99638
 7      90357
 8      89985
 9      77745
 10    736120
Name: count, dtype: int64


In [11]:
# mths_since_last_delinq / mths_since_last_record are null when the borrower has NO
# delinquency / public record on file — the absence itself is the signal, so capture it
# as a flag. Then fill the numeric columns with the median because logistic regression
# rejects NaN; the median-filled version also works fine for tree models.

for col, flag in [('mths_since_last_delinq', 'has_delinq_history'),
                  ('mths_since_last_record', 'has_public_record')]:
    df_lending_club[flag] = df_lending_club[col].notna().astype(int)
    df_lending_club[col] = df_lending_club[col].fillna(df_lending_club[col].median())

print(f"has_delinq_history: {df_lending_club['has_delinq_history'].mean():.1%} of borrowers")
print(f"has_public_record: {df_lending_club['has_public_record'].mean():.1%} of borrowers")

has_delinq_history: 49.1% of borrowers
has_public_record: 16.1% of borrowers


In [12]:
# tot_cur_bal / tot_coll_amt are null when the bureau pull did not report total
# balances (~3.3% of 2012-2015 loans; reported for all 2016-2018 loans). Same pattern
# as the mths_* columns above: capture the absence as a flag, then median-fill so
# logistic regression accepts the matrix.
# collections_12_mths_ex_med has no nulls in the resolved 2012-2018 vintages; any
# residual null is filled with 0 (its overwhelmingly common value) as a safeguard.

for col, flag in [('tot_cur_bal', 'tot_cur_bal_missing'), ('tot_coll_amt', 'tot_coll_amt_missing')]:
    df_lending_club[flag] = df_lending_club[col].isna().astype(int)
    df_lending_club[col] = df_lending_club[col].fillna(df_lending_club[col].median())

n_coll_null = df_lending_club['collections_12_mths_ex_med'].isnull().sum()
df_lending_club['collections_12_mths_ex_med'] = df_lending_club['collections_12_mths_ex_med'].fillna(0)

print(f"tot_cur_bal_missing: {df_lending_club['tot_cur_bal_missing'].mean():.1%} of borrowers")
print(f"tot_coll_amt_missing: {df_lending_club['tot_coll_amt_missing'].mean():.1%} of borrowers")
print(f'collections_12_mths_ex_med residual nulls filled with 0: {n_coll_null:,}')

tot_cur_bal_missing: 0.9% of borrowers
tot_coll_amt_missing: 0.9% of borrowers
collections_12_mths_ex_med residual nulls filled with 0: 0


In [13]:
# Sanity check: the cleaned frame should have no nulls left anywhere

print(f'rows: {len(df_lending_club):,}, columns: {len(df_lending_club.columns)}')
print(f'nulls remaining: {df_lending_club.isnull().sum().sum()}')
df_lending_club.isnull().sum()[lambda s: s > 0]

rows: 2,207,228, columns: 43
nulls remaining: 0


Series([], dtype: int64)

## Target construction

`loan_default = 1` for terminal bad outcomes (Charged Off, Default), `0` for Fully Paid.
Loans still in flight (Current, Late, In Grace Period) have an unknown final status —
labeling them either way would poison the target, so they are excluded from the
modeling dataset.

In [14]:
# Build the binary target: did the borrower default on this loan?
#   loan_default = 1 -> Charged Off or Default (terminal bad outcomes)
#   loan_default = 0 -> Fully Paid (loan repaid in full)
# Unresolved outcomes (Current, In Grace Period, Late 16-30 / 31-120 days) are EXCLUDED:
# their final status is unknown, so labeling them 0 or 1 would poison the target.

target_map = {'Fully Paid': 0, 'Charged Off': 1, 'Default': 1}

df_lending_club['loan_default'] = df_lending_club['loan_status'].map(target_map)

unresolved = df_lending_club['loan_default'].isnull()
print(f'unresolved loans excluded: {unresolved.sum():,}')

df_lending_club = df_lending_club[~unresolved].reset_index(drop=True)
df_lending_club['loan_default'] = df_lending_club['loan_default'].astype(int)

print(f'rows: {len(df_lending_club):,}')
print(f"default rate: {df_lending_club['loan_default'].mean():.1%}")
print(df_lending_club['loan_status'].value_counts())

unresolved loans excluded: 910,330
rows: 1,296,898
default rate: 20.2%
loan_status
Fully Paid     1035418
Charged Off     261440
Default             40
Name: count, dtype: int64


In [15]:
# Save the cleaned, target-ready dataset to Unity Catalog so the feature-engineering /
# model-development notebook can pick it up from a single, stable location.
# (Serverless cannot write data files to /Workspace paths, so a UC table is the persistence layer.)

df_clean_spark = spark.createDataFrame(df_lending_club)
df_clean_spark.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.default.lending_club_cleaned')

print(f'rows written: {len(df_lending_club):,}')
print(f'columns written: {len(df_lending_club.columns)}')

rows written: 1,296,898
columns written: 44


## Result

**1,296,898 resolved loans (≈20% default rate), zero nulls, ready for feature
engineering**, persisted as `workspace.default.lending_club_cleaned`.

Row accounting: 2,260,701 raw rows → 2,207,228 after dropping unsalvageable rows
(−53,473) → 1,296,898 after excluding unresolved loan outcomes (−910,330).